# PromptOps Arena - GRPO Training Notebook

Trains a 1.5B prompt-engineer agent (Qwen2.5-1.5B-Instruct + LoRA) with GRPO
to write better system prompts for a *frozen* 0.5B LLM-under-test, judged by
programmatic verifiers across math / code / JSON tasks.

**OpenEnv Hackathon submission** - Team Dar3devil.

- Space: https://huggingface.co/spaces/Dar3devil/promptops-arena
- Adapter (trained): https://huggingface.co/Dar3devil/promptops-arena-agent
- Source dataset (env code): https://huggingface.co/datasets/Dar3devil/promptops-arena-src
- GitHub: https://github.com/Aarya01Patil/promptops_arena

## How to run

1. Open in Colab (`Runtime` -> `Change runtime type` -> GPU, ideally A100/L4/T4).
2. Run cells top-to-bottom. The full env code is pulled from the public HF dataset.
3. To reproduce the submitted run, leave defaults: 300 steps, batch 8, G=4 (~25 min on H200, ~50 min on A100, ~2 h on T4).

Set `STEPS=20` for a fast smoke run that still produces a real reward curve.

## 1. Install dependencies (TRL 0.21 GRPO stack)

In [ ]:
!pip install -q --upgrade pip
!pip install -q \
    "trl==0.21.0" \
    "transformers==4.55.4" \
    "peft==0.15.2" \
    "accelerate==1.7.0" \
    "datasets==3.6.0" \
    "huggingface_hub>=0.25.0" \
    "jsonschema>=4.20.0" \
    "openenv-core>=0.1.0" \
    "fastapi>=0.110.0" \
    "uvicorn>=0.27.0" \
    "pydantic>=2.0.0" \
    "matplotlib>=3.7.0"

## 2. Pull environment source from public HF dataset

The OpenEnv environment (verifiers, tasks, server) is mirrored to a public HF dataset so this notebook is self-contained.

In [ ]:
import os, sys
from huggingface_hub import snapshot_download

src_dir = snapshot_download(
    repo_id="Dar3devil/promptops-arena-src",
    repo_type="dataset",
    local_dir="/content/promptops-arena",
)
os.chdir(src_dir)
sys.path.insert(0, src_dir)
print("working dir:", os.getcwd())
!ls

## 3. Sanity check the environment + LLM-under-test

In [ ]:
os.environ["PROMPTOPS_LLM_BACKEND"] = "transformers"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from src.envs.promptops_arena.server.environment import PromptOpsArenaEnvironment
from src.envs.promptops_arena.tasks import load_tasks

tasks = load_tasks(split="train")
print(f"loaded {len(tasks)} train tasks")
for t in tasks[:3]:
    print(f"  [{t['type']:5s}] {t['id']}: {t['question'][:70]}")

env = PromptOpsArenaEnvironment(split="train", seed=0)
demo_task = tasks[0]
demo_prompt = "Solve this:"
res = env.execute_prompt(demo_task, demo_prompt)
print("\nzero-shot reward on first task:", res["reward"])

## 4. Configure + launch GRPO training

Tweak `STEPS` for a quicker run. The submitted Adapter on HF Hub was trained with `STEPS=300, BATCH=8, NUM_GENS=4` on a single H200.

In [ ]:
STEPS = 300
BATCH = 8
NUM_GENS = 4
OUT_DIR = "outputs/grpo-lora"
LOG_PATH = "results/training_log.jsonl"

!python scripts/train_grpo.py \
    --steps {STEPS} \
    --batch {BATCH} \
    --num-generations {NUM_GENS} \
    --out {OUT_DIR} \
    --log {LOG_PATH}

## 5. Plot the reward curve from this run

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

rows = [json.loads(l) for l in Path(LOG_PATH).read_text().splitlines() if l.strip()]
rewards = [r["reward"]["total"] for r in rows]
correctness = [r["reward"]["correctness"] for r in rows]

win = max(1, len(rewards) // 30)
ma = [sum(rewards[max(0, i-win):i+1]) / max(1, min(i+1, win)) for i in range(len(rewards))]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rewards, alpha=0.25, label="per-step reward")
ax.plot(ma, lw=2, label=f"moving avg (window={win})")
ax.set_xlabel("reward call (≈ step × num_gens)")
ax.set_ylabel("total reward")
ax.set_title("GRPO training reward curve - PromptOps Arena")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("docs/reward_curve.png", dpi=150)
plt.show()
print(f"n calls: {len(rewards)}, mean reward: {sum(rewards)/len(rewards):.3f}, correct fraction: {sum(correctness)/len(correctness):.3f}")

## 6. Evaluate trained adapter on held-out test split

In [ ]:
!python scripts/eval_trained.py \
    --adapter {OUT_DIR} \
    --per-type 4 \
    --out results/trained_agent.json \
    --max-turns 2

import json
trained = json.load(open("results/trained_agent.json"))
print("by type:", trained["by_type"])
print("overall:", trained["overall"])

## 7. (Optional) push trained adapter to your HF Hub repo

Set `HF_TOKEN` (Colab `Secrets` panel) before running.

In [ ]:
# from huggingface_hub import HfApi, create_repo
# REPO = "<your-username>/promptops-arena-agent"
# create_repo(REPO, repo_type="model", exist_ok=True, private=False)
# HfApi().upload_folder(folder_path=OUT_DIR, repo_id=REPO, repo_type="model",
#                       commit_message="GRPO LoRA adapter")
# HfApi().upload_file(path_or_fileobj=LOG_PATH, path_in_repo="training_log.jsonl",
#                     repo_id=REPO, repo_type="model", commit_message="reward log")

## What this notebook proves

- **Real GRPO loop**, not a stub: every reward comes from the *frozen 0.5B LLM-under-test* generating an answer to a real task, scored by a programmatic verifier (regex/exec/jsonschema).
- **Reward signal grounded in another model's behavior** - which is what makes the *agent* learn how to instruct (transferable across math/code/JSON), rather than how to answer.
- The reward curve in cell 5 is the actual training trajectory; the eval in cell 6 reproduces the headline `10/12` test-split score.